In [1]:
import struct

def bin_to_float64(b: str) -> float:
    """Конвертирует 64-битную бинарную строку в float64"""
    if len(b) != 64:
        raise ValueError("Требуется 64-битная строка")
    
    # Конвертируем бинарную строку в байты (big-endian)
    byte_data = int(b, 2).to_bytes(8, byteorder='big')
    
    # Распаковываем в double
    return struct.unpack('>d', byte_data)[0]

def float64_to_bin(value: float) -> str:
    """Конвертирует float64 в 64-битную бинарную строку"""
    # Упаковываем float64 в байты (big-endian)
    byte_data = struct.pack('>d', value)
    
    # Конвертируем в 64-битное целое
    uint64 = struct.unpack('>Q', byte_data)[0]
    
    # Форматируем в бинарную строку с ведущими нулями
    return bin(uint64)[2:].zfill(64)

In [7]:
def text_to_bits(text, encoding='ascii', errors='ignore'):
    """Преобразует текст в битовую строку."""
    bits = bin(int.from_bytes(text.encode(encoding, errors), 'big'))[2:]
    return bits.zfill(8 * ((len(bits) + 7) // 8))

def text_from_bits(bits, encoding='ascii', errors='ignore'):
    """Преобразует битовую строку в текст."""
    n = int(bits, 2)
    return n.to_bytes((n.bit_length() + 7) // 8, 'big').decode(encoding, errors) or '\0'

def encode_value(vr, value):
    """Кодирует значение в биты согласно VR"""
    try:
        if vr == 'DA':  # Date
            # Проверка формата: YYYYMMDD (8 символов)
            return text_to_bits(value)  # ASCII

        # Добавьте другие VR по мере необходимости
        else:
            raise ValueError(f"Unsupported VR: {vr}")
    
    except Exception as e:
        raise ValueError(f"Не удалось закодировать {vr} {value}: {str(e)}")

def decode_value(vr, value_bits):
    """Декодирует значение из битовой строки согласно VR"""
    try:
        if vr == 'DA':  # Date (ASCII, формат YYYYMMDD)
            # Декодируем как ASCII и обрезаем до 8 символов
            date_str = text_from_bits(value_bits, encoding='ascii', errors='ignore')
            return date_str

        # Добавьте другие VR по мере необходимости
        else:
            raise ValueError(f"Unsupported VR: {vr}")
    
    except Exception as e:
        raise ValueError(f"Ошибка декодирования {vr}: {str(e)}")

# Тест для DA
def test_da():
    """Тестирование кодирования и декодирования для VR DA"""
    test_cases = [
        {'value': '202503201', 'expected': '202503201'},  # Корректная дата
        {'value': '1999123122', 'expected': '1999123122'},  # Корректная дата
        {'value': '20240229', 'expected': '20240229'},  # Корректная дата (високосный год)
    ]

    for case in test_cases:
        value = case['value']
        expected = case['expected']

        # Кодирование
        try:
            bits = encode_value('DA', value)
        except Exception as e:
            print(f"Ошибка кодирования: {str(e)}")
            continue

        # Декодирование
        try:
            decoded_value = decode_value('DA', bits)
        except Exception as e:
            print(f"Ошибка декодирования: {str(e)}")
            continue

        # Проверка
        if decoded_value == expected:
            print(f"Успех: {value} -> {decoded_value}")
        else:
            print(f"Ошибка: {value} -> {decoded_value} (ожидалось {expected})")

# Запуск теста
test_da()

Успех: 202503201 -> 202503201
Успех: 1999123122 -> 1999123122
Успех: 20240229 -> 20240229


In [3]:
# Тест с числом 1.5
num = 1234.5
binary = float64_to_bin(num)  # '0011111111111000000000000000000000000000000000000000000000000000'
restored = bin_to_float64(binary)

print(f"Original: {num}")     # Original: 1.5
print(f"Restored: {restored}")# Restored: 1.5

Original: 1234.5
Restored: 1234.5


In [ ]:
int(,2)

22